<a href="https://colab.research.google.com/github/Anthei0774/Advent-of-Code/blob/main/2018/---%20Day%2016%3A%20Chronal%20Classification%20---.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
puzzle = '''Before: [3, 2, 1, 1]
9 2 1 2
After:  [3, 2, 2, 1]'''
with open('16.txt') as f: puzzle = f.read()
puzzle = puzzle.split('\n\n\n\n')
assert len(puzzle) in [1, 2]

################################################################################
# PART 1

samples = puzzle[0]
samples
samples = samples.split('\n')
samples = [ s for s in samples if s != '' ]
samples

# restructure samples, need constant size for in-place operations
N = len(samples)
i = 0
while i < N:

    s = samples[: 3]
    assert len(s) == 3

    s[0] = s[0][9 : -1].split(', ')
    s[0] = list(map(int, s[0]))

    s[1] = s[1].split(' ')
    s[1] = list(map(int, s[1]))

    s[2] = s[2][9 : -1].split(', ')
    s[2] = list(map(int, s[2]))

    samples = samples[3 :]
    samples.append({ 'before' : s[0], 'operation' : s[1], 'after' : s[2], 'potentially' : [] })

    i += 3

samples

# roll through each sample and operator
operators = ['addr', 'addi', 'mulr', 'muli', 'banr', 'bani', 'borr', 'bori', 'setr', 'seti', 'gtir', 'gtri', 'gtrr', 'eqir', 'eqri', 'eqrr']
for s in samples:
    for op in operators:
        result = None

        if op in ['addr', 'addi', 'mulr', 'muli', 'banr', 'bani', 'borr', 'bori']:
            a = s['before'][ s['operation'][1] ]
            b = s['operation'][2] if op[3] == 'i' else s['before'][ s['operation'][2] ]
            c = 0
            if op[:3] == 'add': c = a + b
            elif op[:3] == 'mul': c = a * b
            elif op[:3] == 'ban': c = a & b
            else: c = a | b
            i = s['operation'][3]
            result = s['before'].copy()
            result[i] = c
        if op in ['setr', 'seti']:
            a = s['operation'][1] if op[3] == 'i' else s['before'][ s['operation'][1] ]
            i = s['operation'][3]
            result = s['before'].copy()
            result[i] = a
        if op in ['gtir', 'gtri', 'gtrr', 'eqir', 'eqri', 'eqrr']:
            a = s['operation'][1] if op[2] == 'i' else s['before'][ s['operation'][1] ]
            b = s['operation'][2] if op[3] == 'i' else s['before'][ s['operation'][2] ]
            c = 0
            if op[:2] == 'gt': c = a > b
            else: c = a == b
            i = s['operation'][3]
            result = s['before'].copy()
            result[i] = c

        # if tmp result matches samples result, then op can be matched with the opcode
        if result == s['after']:
            s['potentially'].append(op)

samples
len([ s for s in samples if len(s['potentially']) >= 3 ])

################################################################################
# PART 2

operators = { op : [] for op in operators }

# assign opcodes to operators (= reversing the previous part)
for s in samples:
    for op in s['potentially']:
        if s['operation'][0] not in operators[op]:
            operators[op].append( s['operation'][0] )
operators

# reduce operators mapping
opcode_not_found = list(operators)
while True:

    # search those operators which do not have opcodes yet
    for op in opcode_not_found:

        # found one with obvious relationship
        if len(operators[op]) == 1:

            # reduce list to single integer
            opcode = operators[op][0]
            operators[op] = opcode
            opcode_not_found.remove(op)

            # remove that opcode from all other possibilities
            for opp in operators:
                if op != opp and type(operators[opp]) == list:
                    if opcode in operators[opp]:
                        operators[opp].remove(opcode)

    # exit is nothing to be processed (= done)
    if opcode_not_found == []:
        break

# reverse 1-1 mapping and sort
operators = { operators[op] : op for op in operators }
operators = { opcode : operators[opcode] for opcode in sorted(operators) }
operators


# process / format tests
if len(puzzle) == 2: test = puzzle[1]
test = test.split('\n')
test = [ line.split(' ') for line in test ]
test = [ list(map(int, line)) for line in test ]
test = [ [ operators[ line[0] ] ] + line[1 : ] for line in test ]
test


# RUN BIG CHUNGUS AGAIN
registers = [0, 0, 0, 0]
for t in test:
    # print(registers, t)
    op = t[0]

    if op in ['addr', 'addi', 'mulr', 'muli', 'banr', 'bani', 'borr', 'bori']:
        a = registers[ t[1] ]
        b = t[2] if op[3] == 'i' else registers[ t[2] ]
        c = 0
        if op[:3] == 'add': c = a + b
        elif op[:3] == 'mul': c = a * b
        elif op[:3] == 'ban': c = a & b
        else: c = a | b
        registers[ t[3] ] = c
    if op in ['setr', 'seti']:
        a = t[1] if op[3] == 'i' else registers[ t[1] ]
        registers[ t[3] ] = a
    if op in ['gtir', 'gtri', 'gtrr', 'eqir', 'eqri', 'eqrr']:
        a = t[1] if op[2] == 'i' else registers[ t[1] ]
        b = t[2] if op[3] == 'i' else registers[ t[2] ]
        c = 0
        if op[:2] == 'gt': c = a > b
        else: c = a == b
        registers[ t[3] ] = c

registers[0]